# 01 · SFT через LoRA

**Цель:** модель на каждый запрос вызывает нужный навык и отвечает по правилам продукта. Обучаем на 76 траекториях трейн-сета (другие работы), меряем на 36 ситуациях голд-сета.

**Схема:** замер до → адаптер → обучение → замер после; те же четыре ситуации печатаются целиком. Математика — `books/02-sft-math.pdf`. Все вызовы `peft` и `transformers` на виду.

In [ ]:
from common import (MODEL_ID, SYSTEM, TOOLS, RUNS, SHOWCASE, load_rows, trajectory, evaluate, judge, judge_rate,
                    fmt, table, show_case)

import json
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model
from vlmkit import ChatCollator, memory_report, preview, evaluate as ev
from vlmkit.compat import supported, first_accepted

golden = load_rows("golden")
train = load_rows("train")

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map={"": 0},
    attn_implementation="sdpa", trust_remote_code=True,
)
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True, max_pixels=1003520)
print(memory_report())

## До

In [ ]:
def report(results, per_row, verdicts=None, ids=SHOWCASE):
    """Полные ответы модели на показательные ситуации с проверками и вердиктом судьи."""
    for i, row in enumerate(golden):
        if row["id"] in ids:
            show_case(row, results[i], per_row[i]["checks"], verdicts[i] if verdicts else None)

before, before_rows, before_summary = evaluate(model, processor, golden)
before_verdicts = judge(model, processor, golden, [r["text"] for r in before])
before_summary["judge_pass"] = judge_rate(before_verdicts)
print(fmt(before_summary), f"судья {before_summary['judge_pass']:.0%}")
report(before, before_rows, before_verdicts)

## Данные

Траектория — четыре реплики: запрос с документом, вызов `select_skill`, текст навыка, ответ. Открыты для градиента вызов и ответ; всё остальное скрыто. Perplexity эталонов голд-сета считается по тем же позициям — это отложенная выборка, документы в ней другие.

In [ ]:
train_samples = [trajectory(r) for r in train]
held = [trajectory(r) for r in golden]

print(len(train_samples), "траекторий; по навыкам:", {s: sum(r["skill"] == s for r in train) for s in sorted({r["skill"] for r in train})})
print(preview(train_samples[0], processor, system=SYSTEM, tools=TOOLS)[-1500:])

ppl_before = ev.perplexity(model, processor, held, system=SYSTEM, tools=TOOLS)
print(f"\nperplexity эталонов голд-сета до обучения: {ppl_before:.2f}")

## Адаптер

Три решения в конфиге, которые не косметика: регекс с отрицательным просмотром выкидывает визуальную башню; `use_rslora=True` меняет масштаб с `α/r` на `α/√r`; `r=16` — правила поведения низкоранговые, это изменение манеры, а не новый навык.

In [ ]:
lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=r"^(?!.*(visual|vision)).*(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)$",
    use_rslora=True,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora)
model.print_trainable_parameters()

# Без этого при gradient checkpointing градиент не доходит до адаптеров.
# Ошибки не будет — обучение просто ничего не даст.
model.enable_input_require_grads()
model.config.use_cache = False

## Обучение

Коллатору передаётся тот же `SYSTEM` и та же схема `TOOLS`, что при генерации: модель учится ровно на том промпте, с которым будет работать. `remove_unused_columns=False` обязательно, иначе `Trainer` выбросит из батча всё, чего нет в сигнатуре `forward`.

In [ ]:
collator = ChatCollator(processor, system=SYSTEM, tools=TOOLS)

args = dict(
    output_dir=str(RUNS / "sft"),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="adamw_torch_fused",
    logging_steps=5,
    save_strategy="no",
    remove_unused_columns=False,
    report_to=[],
    seed=42,
)
args |= first_accepted(TrainingArguments, {"warmup_ratio": 0.05, "warmup_steps": 2})

trainer = Trainer(
    model=model,
    args=TrainingArguments(**supported(TrainingArguments, args)),
    train_dataset=train_samples,
    data_collator=collator,
)
result = trainer.train()
print(f"loss {result.training_loss:.3f}, {result.metrics.get('train_runtime', 0)/60:.1f} мин")

## После

Те же 36 ситуаций, тот же судья (базовая модель, адаптер на время суда отключается). Смотрите сначала на четыре полных ответа, потом на таблицу.

In [ ]:
model.eval()
after, after_rows, after_summary = evaluate(model, processor, golden)
after_verdicts = judge(model, processor, golden, [r["text"] for r in after])
after_summary["judge_pass"] = judge_rate(after_verdicts)
report(after, after_rows, after_verdicts)

ppl_after = ev.perplexity(model, processor, held, system=SYSTEM, tools=TOOLS)
print(f"\nperplexity эталонов: {ppl_before:.2f} → {ppl_after:.2f}")
table({"база": before_summary, "SFT": after_summary})

print("\nпо проверкам (база → SFT):")
for k in sorted(k for k in after_summary if k.startswith("check[")):
    print(f"  {k:24} {before_summary.get(k, 0):5.0%} → {after_summary[k]:5.0%}")

model.save_pretrained(str(RUNS / "sft"))
(RUNS / "sft.json").write_text(json.dumps(after_summary, ensure_ascii=False, indent=2), encoding="utf-8")

## Адаптер как переключатель

Исходные веса не менялись: с выключенным адаптером тот же процесс отвечает как базовая модель.

In [ ]:
with model.disable_adapter():
    print("адаптер выключен:", fmt(evaluate(model, processor, golden)[2]))
print("адаптер включён: ", fmt(after_summary))

## На что смотреть

**skill_acc вырос, checks_all нет** — модель выучила вызов, но не манеру ответа: смотрите, какие проверки проваливаются; чаще всего `few_questions` и `ends_step` — ответ длинный и без шага. Помогает больше эпох или больший ранг.

**refusal FPR вырос** — переобобщение отказов: модель отказывает и на легитимных запросах. Больше примеров без отказа в трейне.

**judge_pass не растёт при росте автопроверок** — содержательные пункты: судья видит, что модель формулирует гипотезу за студента, хотя форма правильная. Здесь помогают пары в `03-dpo`: плохой ответ показывает границу явно.